# XENON PHONON DISPERSION AT 10 K

## DATA

Resolution-corrected phonon energies measured by neutron inelastic scattering at 10 K, transcribed from Table I of N. A. Lurie, G. Shirane, and J. Skalyo, Jr., [*Phonon dispersion relations in xenon at 10 K*](https://doi.org/10.1103/PhysRevB.9.5300), *Physical Review B* **9**, 5300–5306 (1974). The local [data extract](../data/lurie1974/phonon_energies.csv) retains every tabulated principal-direction measurement. `xi` is the reduced wave vector, and `sigma_mev` is the authors' reported two-standard-error uncertainty. The open Γ–L points are digitized from Palmer *et al.* (1974). The continuous curves evaluate the paper's Model 2 third-neighbor Born–von Kármán fit.

In [1]:
from itertools import permutations, product
from pathlib import Path

import numpy as np

repo = Path("..") if Path.cwd().name == "examples" else Path(".")
data = np.genfromtxt(
    repo / "data/lurie1974/phonon_energies.csv", delimiter=",", names=True,
    dtype=[("direction", "U3"), ("branch", "U6"), ("xi", "f8"), ("energy_mev", "f8"), ("sigma_mev", "f8")],
)
palmer = np.genfromtxt(
    repo / "data/lurie1974/palmer_4k_gamma_l_digitized.csv", delimiter=",", names=True
)

# Model 2 of Table II. Vectors are in units of a/2; force constants in dyn cm⁻¹.
SHELLS = (
    (np.array((1, 1, 0)), np.array(((913, 934, 0), (934, 913, 0), (0, 0, -5)), dtype=float)),
    (np.array((2, 0, 0)), np.array(((-76, 0, 0), (0, 12, 0), (0, 0, 12)), dtype=float)),
    (np.array((2, 1, 1)), np.array(((-7, -3, -3), (-3, 0, -9), (-3, -9, 0)), dtype=float)),
)
LATTICE_PARAMETER_M = 6.129e-10
XENON_MASS_KG = 131.293 * 1.66053906660e-27
HBAR_J_S = 1.054571817e-34
MEV_J = 1.602176634e-22


def equivalent_neighbors(vector, force):
    seen, neighbors = set(), []
    for order in permutations(range(3)):
        for signs in product((-1, 1), repeat=3):
            rotation = np.zeros((3, 3))
            rotation[range(3), order] = signs
            image = tuple((rotation @ vector).astype(int))
            if image not in seen:
                seen.add(image)
                neighbors.append((np.asarray(image), rotation @ force @ rotation.T))
    return neighbors


NEIGHBORS = tuple(
    (vector * LATTICE_PARAMETER_M / 2, force * 1e-3)
    for reference, matrix in SHELLS
    for vector, force in equivalent_neighbors(reference, matrix)
)


def energies_mev(q_reduced):
    """Return energy-ordered phonon energies at q in units of 2π/a."""
    q = np.asarray(q_reduced, dtype=float)
    dynamical = np.zeros((3, 3))
    for vector, force in NEIGHBORS:
        phase = 2 * np.pi * np.dot(q, vector / LATTICE_PARAMETER_M)
        dynamical += force * (1 - np.cos(phase))
    omega = np.sqrt(np.clip(np.linalg.eigvalsh(dynamical) / XENON_MASS_KG, 0, None))
    return HBAR_J_S * omega / MEV_J


## PLOT


In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import lab74


def model_path(start, end, x_start, x_end, samples=401):
    fraction = np.linspace(0, 1, samples)
    q = start + fraction[:, None] * (end - start)
    return np.linspace(x_start, x_end, samples), np.array([energies_mev(point) for point in q])


def observations(direction, branch, x):
    rows = data[(data["direction"] == direction) & (data["branch"] == branch)]
    ax.plot(x(rows["xi"]), rows["energy_mev"], color=lab74.INK, linestyle="none", marker="o", markersize=3.4, markerfacecolor=lab74.INK, markeredgecolor=lab74.INK, zorder=3)


lab74.use(accent=None, annotation_face="gothic")
plt.rcParams.update({"font.size": 9.5, "axes.labelsize": 10, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5})
out = repo / "examples/output"
out.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(5.5, 3.0))
fig.subplots_adjust(left=0.09, right=0.985, bottom=0.18, top=0.92)

# Horizontal distances follow the reciprocal-space path lengths: 1 : 1 : √2 : √3/2.
x_gamma_x, x_x_edge = 1.0, 2.0
x_gamma, x_l = x_x_edge + np.sqrt(2), x_x_edge + np.sqrt(2) + np.sqrt(3) / 2

# The solid curves are the paper's Model 2 third-neighbor Born-von Kármán fit.
paths = (
    (np.array((0, 0, 0)), np.array((1, 0, 0)), 0, 1),
    (np.array((1, 0, 0)), np.array((1, 1, 0)), 1, 2),
    (np.array((1, 1, 0)), np.array((0, 0, 0)), x_x_edge, x_gamma),
    (np.array((0, 0, 0)), np.array((0.5, 0.5, 0.5)), x_gamma, x_l),
)
for start, end, x0, x1 in paths:
    x, energy = model_path(start, end, x0, x1)
    for mode in energy.T:
        ax.plot(x, mode, color=lab74.INK, linewidth=1.1, linestyle="-", marker="None", zorder=1)

for name in ("T", "L"):
    observations("100", name, lambda xi: xi)
for name in ("Lambda", "Pi"):
    observations("110", name, lambda xi: 1 + xi)
for name in ("T1", "T2", "L"):
    observations("110", name, lambda xi: x_gamma - np.sqrt(2) * xi)
for name in ("T", "L"):
    observations("111", name, lambda xi: x_gamma + np.sqrt(3) * xi)

# Palmer et al.'s four digitized 4 K Γ–L points are open circles.
lab74.errorbar(ax, x_gamma + np.sqrt(3) * palmer["xi"], palmer["energy_mev"], palmer["sigma_mev"], linestyle="none", markersize=3.0, markerfacecolor=lab74.PAPER, markeredgewidth=0.6, zorder=4)

for location in (x_gamma_x, x_x_edge, x_gamma):
    ax.axvline(location, color=lab74.INK, linewidth=0.6, zorder=0)
    for energy in range(1, 7):
        ax.plot((location - 0.045, location + 0.045), (energy, energy), color=lab74.INK, linewidth=0.6, linestyle="-", marker="None", zorder=0)

ax.set(xlim=(0, x_l), ylim=(0, 6.6), ylabel="ENERGY (meV)")
# Every requested x position is a major tick; blank labels provide the 0.1 subdivisions.
x_ticks = (
    (np.arange(0, 1, 0.1), ("0", "", "0.2", "", "0.4", "", "0.6", "", "0.8", "")),
    (1 + np.arange(0, 1.01, 0.1), ("0", "", "0.2", "", "0.4", "", "0.4", "", "0.2", "", "0")),
    (x_gamma - np.sqrt(2) * np.arange(0.9, -0.01, -0.1), ("", "0.8", "", "0.6", "", "0.4", "", "0.2", "", "0")),
    (x_gamma + np.sqrt(3) * np.arange(0.1, 0.51, 0.1), ("0.1", "0.2", "0.3", "0.4", "0.5")),
)
ax.xaxis.set_major_locator(mticker.FixedLocator(np.concatenate([locations for locations, _ in x_ticks])))
ax.xaxis.set_major_formatter(mticker.FixedFormatter([label for _, labels in x_ticks for label in labels]))
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_locator(mticker.MultipleLocator(1))
ax.yaxis.set_minor_locator(mticker.NullLocator())
lab74.format_frame(ax, style="closed")
ax.tick_params(axis="x", which="major", length=6.5, width=0.55, labelbottom=True)

sigma_x = x_x_edge + np.sqrt(2) / 2
lambda_x = x_gamma + np.sqrt(3) / 4
k_labels = ((0, "Γ"), (0.5, "Δ"), (x_gamma_x, "X"), (1.15, "Z"), (1.5, "W"), (1.85, "Z"), (x_x_edge, "X"), (sigma_x, "Σ"), (x_gamma, "Γ"), (lambda_x, "Λ"), (x_l, "L"))
for x, label in k_labels:
    ax.text(x, 1.005, label, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=plt.rcParams["axes.labelsize"])
for start, end in ((0.56, 0.72), (1.21, 1.37), (1.79, 1.63), (sigma_x - 0.06, sigma_x - 0.22), (lambda_x + 0.06, lambda_x + 0.22)):
    lab74.axis_arrow(ax, start, end, offset=0.035)
for start, end in ((0.36, 0.64), (1.10, 1.35), (1.90, 1.64), (2.857, 2.557), (x_gamma + 0.27, x_gamma + 0.55)):
    lab74.axis_arrow(ax, start, end, side="bottom", offset=0.085)

for x, label in ((0.5, "[100]"), ((x_x_edge + x_gamma) / 2, "[110]"), ((x_gamma + x_l) / 2, "[111]")):
    ax.text(x, 0.93, label, transform=ax.get_xaxis_transform(), ha="center", va="top")
ax.text(1.5, 1.95, "XENON\n10K", ha="center", va="center")

for x, y, label in ((0.42, 4.24, "L"), (0.69, 2.75, "T"), (1.50, 5.14, "Π"), (1.50, 3.28, "Λ"), (x_gamma - np.sqrt(2) * 0.553, 2.23, "T₁"), (x_gamma - np.sqrt(2) * 0.514, 3.28, "T₂"), (x_gamma - np.sqrt(2) * 0.287, 3.94, "L"), (x_gamma + np.sqrt(3) * 0.189, 3.67, "L"), (x_gamma + np.sqrt(3) * 0.365, 1.91, "T")):
    ax.text(x, y, label, ha="center", va="center")

fig.savefig(out / "07_xenon_phonons.png", dpi=300)
plt.close(fig)
